In [25]:
import os
import subprocess
import time
import shutil
import csv
from datetime import datetime
import stat

import re

import ast
import json
import shutil
import builtins

In [16]:
# KONFIGURASI

input_file = r"D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\asli\nim_github(10).txt"
projects_folder = r"D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\code_mhs"
os.makedirs(projects_folder, exist_ok=True)

In [17]:
#LOG Clone

log_folder = r"D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\log"
os.makedirs(log_folder, exist_ok=True)

log_file = os.path.join(log_folder, "log_clone.csv")

# SETUP LOG

if not os.path.exists(log_file):
    with open(log_file, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["timestamp", "nim", "url", "status", "message"])

def write_log(nim, url, status, message):
    with open(log_file, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            nim,
            url,
            status,
            message
        ])

# Pengumpulan Data

In [18]:
# BACA DATASET
# dataset berupa nim | url
def load_dataset(file_path):

    dataset = []

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            parts = line.split("|")

            if len(parts) != 2:
                continue

            nim = parts[0].strip()
            url = parts[1].strip()

            dataset.append((nim, url))

    return dataset

In [19]:
# HANYA SIMPAN .py & .ipynb

def keep_only_code_files(repo_path):

    for root, dirs, files in os.walk(repo_path):

        # Jangan masuk folder .git
        if ".git" in dirs:
            dirs.remove(".git")

        for file in files:
            if not (file.endswith(".py") or file.endswith(".ipynb")):
                try:
                    os.remove(os.path.join(root, file))
                except:
                    pass

    # Hapus folder kosong
    for root, dirs, files in os.walk(repo_path, topdown=False):
        if not os.listdir(root):
            try:
                os.rmdir(root)
            except:
                pass


In [20]:
# Remove folder gagal

def remove_readonly(func, path, exc_info):
    """
    Menghapus atribut read-only lalu retry delete.
    """
    try:
        os.chmod(path, stat.S_IWRITE)
        func(path)
    except Exception as e:
        print(f"Gagal paksa hapus: {path}")
        print(f"->   Alasan: {str(e)}")

In [21]:
# CLONE FUNCTION

def clone_repo(nim, url):

    global success_count, fail_count, skip_count

    # Jika NIM kosong
    if not nim:
        print(f"[SKIP] URL tanpa NIM → {url}")
        write_log("", url, "SKIPPED", "NIM kosong")
        skip_count += 1
        return

    # Bersihkan URL jika ada /tree/
    if "/tree/" in url:
        url = url.split("/tree/")[0]

    target_path = os.path.join(projects_folder, nim)

    # Jika sudah pernah clone
    if os.path.exists(target_path):
        print(f"[SKIP] {nim} sudah ada.")
        write_log(nim, url, "SKIPPED", "Folder sudah ada")
        skip_count += 1
        return

    print(f"[CLONE] {url}")

    try:
        subprocess.run(
            ["git", "clone", url, target_path],
            check=True,
            timeout=300,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.PIPE
        )

        # Simpan hanya .py dan .ipynb
        keep_only_code_files(target_path)

        print(f"   ✅ Berhasil clone {nim}")
        write_log(nim, url, "SUCCESS", "Clone berhasil")
        success_count += 1

        time.sleep(1)

    except subprocess.CalledProcessError as e:

        error_msg = e.stderr.decode(errors="ignore")

        print(f"   ❌ Gagal clone {nim}")
        print(f"   Alasan:\n{error_msg}")

        write_log(nim, url, "FAILED", error_msg)

        # Hapus folder jika setengah clone
        if os.path.exists(target_path):
            try:
                shutil.rmtree(target_path, onerror=remove_readonly)
                print("   🧹 Folder clone dihapus")
            except Exception as delete_error:
                print("   ⚠ Gagal hapus folder clone")
                print(str(delete_error))

        fail_count += 1

    except subprocess.TimeoutExpired:

        print(f"   ⏰ Timeout clone {nim}")
        write_log(nim, url, "FAILED", "Timeout")

        if os.path.exists(target_path):
            try:
                shutil.rmtree(target_path)
            except:
                pass

        fail_count += 1

In [22]:
dataset = load_dataset(input_file)

total_url = len(dataset)
success_count = 0
fail_count = 0
skip_count = 0

print(f"\nTotal URL dalam dataset: {total_url}\n")

for nim, url in dataset:
    clone_repo(nim, url)

print("\n===== RINGKASAN CLONING =====")
print(f"Total URL      : {total_url}")
print(f"Berhasil clone : {success_count}")
print(f"Gagal clone    : {fail_count}")
print(f"Skipped        : {skip_count}")
print("================================")
print(f"Log tersimpan di: {log_file}")


Total URL dalam dataset: 10

[CLONE] https://github.com/Katakon17/2241720092_ML_2025
   ✅ Berhasil clone 2241720092
[CLONE] https://github.com/4rdnac/2341720187_ML_2025
   ✅ Berhasil clone 2341720187
[CLONE] https://github.com/fajrulsantoso/244107023010_ML_2025
   ❌ Gagal clone 244107023010
   Alasan:
Cloning into 'D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\code_mhs\244107023010'...
error: invalid path 'JS08 /JS08'
fatal: unable to checkout working tree
You can inspect what was checked out with 'git status'
and retry with 'git restore --source=HEAD :/'


   🧹 Folder clone dihapus
[CLONE] https://github.com/FarrelAD/2341720081_ML_2025
   ✅ Berhasil clone 2341720081
[CLONE] https://github.com/emkafie/Machine-Learning
   ✅ Berhasil clone 2341720176
[CLONE] https://github.com/AstorBoy11/2341720095_ML_2025.git
   ✅ Berhasil clone 2341720095
[CLONE] https://github.com/NathanaelGracedo/2341720217_ML_202525
   ❌ Gagal clone 2341720217
   Alasan:
Cloning into 'D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi

## normalisasi struktur direktori

In [23]:
def remove_empty_folders(base_path):
    """
    Menghapus semua folder kosong kecuali folder utama mahasiswa.
    """
    for root, dirs, files in os.walk(base_path, topdown=False):

        # Jangan hapus root utama mahasiswa
        if root == base_path:
            continue

        if not os.listdir(root):
            try:
                os.rmdir(root)
            except:
                pass

In [24]:
log_file = os.path.join(log_folder, "log_normalisasiStrukturNamaFile.csv")

# ==============================
# SETUP LOG
# ==============================

if not os.path.exists(log_file):
    with open(log_file, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["timestamp", "nim", "action", "detail"])

def write_log(nim, action, detail):
    with open(log_file, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            nim,
            action,
            detail
        ])

# ==============================
# IDENTIFIKASI MODUL DARI NAMA
# ==============================

def identify_module_from_name(name):

    name_upper = name.upper()

    # JSxx
    match_js = re.search(r"JS\s?0?(\d+)", name_upper)
    if match_js:
        return f"m{int(match_js.group(1)):02d}"

    if "KUIS" in name_upper:
        return "kuis"

    if "UTS" in name_upper:
        return "uts"

    if "PBL" in name_upper:
        return "pbl"

    if "KELOMPOK" in name_upper:
        return "kelompok"

    return None

# ==============================
# NORMALISASI FILE
# ==============================

def normalize_file_name(file_name, module_folder, counter_dict):

    name_upper = file_name.upper()
    ext = os.path.splitext(file_name)[1]

    # PRAKTIKUM Pxx
    match_p = re.search(r"\bP\s?0?(\d+)", name_upper)
    if match_p:
        number = int(match_p.group(1))
        return f"p{number:02d}{ext}"

    # TP
    if "TP" in name_upper:
        counter_dict[module_folder] += 1
        return f"tp{counter_dict[module_folder]:02d}{ext}"

    # KUIS / UTS
    if module_folder in ["kuis", "uts", "pbl", "kelompok"]:
        return f"{module_folder}{ext}"

    return file_name

# ==============================
# NORMALISASI REPO
# ==============================

def normalize_repository(nim_path):

    nim = os.path.basename(nim_path)
    print(f"\n🔎 Normalisasi {nim}")

    counter_dict = {}
    unclassified_path = os.path.join(nim_path, "unclassified")
    os.makedirs(unclassified_path, exist_ok=True)

    # =========================
    # STEP 1 - Rename folder level atas
    # =========================
    for folder in os.listdir(nim_path):

        old_folder_path = os.path.join(nim_path, folder)

        if not os.path.isdir(old_folder_path):
            continue

        new_module = identify_module_from_name(folder)

        if new_module:
            new_folder_path = os.path.join(nim_path, new_module)

            if old_folder_path != new_folder_path:
                os.rename(old_folder_path, new_folder_path)
                write_log(nim, "RENAME_FOLDER", f"{folder} → {new_module}")

    # =========================
    # STEP 2 - Pindahkan & rename file
    # =========================
    for root, dirs, files in os.walk(nim_path):

        for file in files:

            if not (file.endswith(".py") or file.endswith(".ipynb")):
                continue

            file_path = os.path.join(root, file)

            # Tentukan modul dari folder atau file
            module = identify_module_from_name(root)
            if not module:
                module = identify_module_from_name(file)

            if not module:
                target_folder = unclassified_path
            else:
                target_folder = os.path.join(nim_path, module)
                os.makedirs(target_folder, exist_ok=True)

            if module not in counter_dict:
                counter_dict[module] = 0

            new_name = normalize_file_name(file, module, counter_dict)
            target_path = os.path.join(target_folder, new_name)

            if file_path != target_path:
                shutil.move(file_path, target_path)
                write_log(nim, "MOVE_RENAME_FILE", f"{file} → {module}/{new_name}")

    # =========================
    # STEP 3 - Hapus folder kosong
    # =========================
    remove_empty_folders(nim_path)

    print(f"   ✅ Selesai {nim}")

# ==============================
# MAIN LOOP
# ==============================

print("\n🚀 Mulai normalisasi struktur & nama file...")

for student in os.listdir(projects_folder):

    student_path = os.path.join(projects_folder, student)

    if os.path.isdir(student_path):
        normalize_repository(student_path)

print("\n===== NORMALISASI SELESAI =====")
print(f"Log tersimpan di: {log_file}")


🚀 Mulai normalisasi struktur & nama file...

🔎 Normalisasi 2241720092
   ✅ Selesai 2241720092

🔎 Normalisasi 2341720081
   ✅ Selesai 2341720081

🔎 Normalisasi 2341720095
   ✅ Selesai 2341720095

🔎 Normalisasi 2341720096
   ✅ Selesai 2341720096

🔎 Normalisasi 2341720117
   ✅ Selesai 2341720117

🔎 Normalisasi 2341720163
   ✅ Selesai 2341720163

🔎 Normalisasi 2341720176
   ✅ Selesai 2341720176

🔎 Normalisasi 2341720187
   ✅ Selesai 2341720187

===== NORMALISASI SELESAI =====
Log tersimpan di: D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\log\log_normalisasiStrukturNamaFile.csv


## preprocessing code

In [28]:
preprocessed_folder = projects_folder + "_preprocessed"
os.makedirs(preprocessed_folder, exist_ok=True)

log_file = os.path.join(log_folder, "log_preprocessing.csv")

# ==============================
# SETUP LOG
# ==============================

if not os.path.exists(log_file):
    with open(log_file, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["timestamp", "file_path", "status", "message"])

def write_log(file_path, status, message):
    with open(log_file, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            file_path,
            status,
            message
        ])

# ==============================
# EKSTRAK NOTEBOOK
# ==============================

def extract_code_from_notebook(path):
    try:
        with open(path, "r", encoding="utf-8") as f:
            notebook = json.load(f)
    except:
        return None

    code_blocks = []

    for cell in notebook.get("cells", []):
        if cell.get("cell_type") != "code":
            continue
        source = "".join(cell.get("source", []))
        code_blocks.append(source)

    return "\n".join(code_blocks)

# ==============================
# VALIDASI CODE
# ==============================

def validate_code(code):
    if not code or not code.strip():
        return False, "File kosong"

    try:
        ast.parse(code)
        return True, "Valid"
    except SyntaxError as e:
        return False, f"SyntaxError baris {e.lineno}: {e.msg}"
    except Exception as e:
        return False, str(e)

# ==============================
# CLEANING CODE
# ==============================

def basic_cleaning(code):
    """
    Cleaning awal sebelum validasi:
    - Hapus magic command
    - Hapus karakter aneh
    """
    code = re.sub(r"^\s*%.*$", "", code, flags=re.MULTILINE)
    code = re.sub(r"^\s*!.*$", "", code, flags=re.MULTILINE)
    return code

def advanced_cleaning(code):
    """
    Cleaning lanjutan setelah validasi
    - Hapus docstring
    - Hapus komentar
    - Hapus baris kosong berlebih
    """
    try:
        tree = ast.parse(code)
    except:
        return None

    # Hapus docstring
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.ClassDef, ast.Module)):
            if (
                node.body
                and isinstance(node.body[0], ast.Expr)
                and isinstance(node.body[0].value, ast.Str)
            ):
                node.body.pop(0)

    cleaned = ast.unparse(tree)

    # Hapus komentar
    cleaned = re.sub(r"#.*", "", cleaned)

    # Hapus baris kosong
    lines = cleaned.splitlines()
    lines = [line.rstrip() for line in lines if line.strip() != ""]

    return "\n".join(lines)

# ==============================
# NORMALISASI IDENTIFIER
# ==============================

class IdentifierNormalizer(ast.NodeTransformer):

    def __init__(self):
        self.var_map = {}
        self.func_map = {}
        self.arg_map = {}

        self.var_counter = 1
        self.func_counter = 1
        self.arg_counter = 1

        self.builtin_names = set(dir(builtins))

    def visit_FunctionDef(self, node):

        if node.name not in self.func_map:
            self.func_map[node.name] = f"func{self.func_counter}"
            self.func_counter += 1

        node.name = self.func_map[node.name]

        for arg in node.args.args:
            if arg.arg not in self.arg_map:
                self.arg_map[arg.arg] = f"arg{self.arg_counter}"
                self.arg_counter += 1
            arg.arg = self.arg_map[arg.arg]

        self.generic_visit(node)
        return node

    def visit_Name(self, node):

        if node.id in self.builtin_names:
            return node

        if node.id in self.arg_map:
            node.id = self.arg_map[node.id]
            return node

        if node.id not in self.var_map:
            self.var_map[node.id] = f"var{self.var_counter}"
            self.var_counter += 1

        node.id = self.var_map[node.id]
        return node

def normalize_identifiers(code):
    try:
        tree = ast.parse(code)
        normalizer = IdentifierNormalizer()
        tree = normalizer.visit(tree)
        ast.fix_missing_locations(tree)
        return ast.unparse(tree)
    except Exception as e:
        return None

# ==============================
# MAIN PREPROCESSING
# ==============================

print("\n🚀 Mulai preprocessing...")

valid_count = 0
invalid_count = 0
empty_count = 0

for root, dirs, files in os.walk(projects_folder):

    relative_path = os.path.relpath(root, projects_folder)
    target_root = os.path.join(preprocessed_folder, relative_path)
    os.makedirs(target_root, exist_ok=True)

    for file in files:

        if not (file.endswith(".py") or file.endswith(".ipynb")):
            continue

        source_path = os.path.join(root, file)

        # === Ambil kode ===
        if file.endswith(".py"):
            with open(source_path, "r", encoding="utf-8") as f:
                code = f.read()
        else:
            code = extract_code_from_notebook(source_path)

        if code is None:
            write_log(source_path, "FAILED", "Tidak bisa membaca file")
            continue

        # === CLEANING AWAL (magic command dulu) ===
        code = basic_cleaning(code)

        # === VALIDASI ===
        is_valid, reason = validate_code(code)
        if not is_valid:
            write_log(source_path, "INVALID", reason)
            if "kosong" in reason.lower():
                empty_count += 1
            else:
                invalid_count += 1
            continue

        # === CLEANING LANJUTAN ===
        cleaned = advanced_cleaning(code)
        if not cleaned:
            write_log(source_path, "FAILED", "Cleaning gagal")
            continue

        # === NORMALISASI IDENTIFIER ===
        normalized = normalize_identifiers(cleaned)
        if not normalized:
            write_log(source_path, "FAILED", "Identifier normalization gagal")
            continue

        # === SIMPAN ===
        new_filename = file.replace(".ipynb", ".py")
        target_path = os.path.join(target_root, new_filename)

        with open(target_path, "w", encoding="utf-8") as f:
            f.write(normalized)

        write_log(source_path, "SUCCESS", "Preprocessing berhasil")
        valid_count += 1

print("\n===== RINGKASAN PREPROCESSING =====")
print(f"File valid & diproses : {valid_count}")
print(f"File kosong           : {empty_count}")
print(f"File syntax error     : {invalid_count}")
print("====================================")
print(f"Hasil tersimpan di: {preprocessed_folder}")


🚀 Mulai preprocessing...

===== RINGKASAN PREPROCESSING =====
File valid & diproses : 312
File kosong           : 4
File syntax error     : 3
Hasil tersimpan di: D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\code_mhs_preprocessed
